# PLEXA cascading routing demo

This example uses synthetic road geometries, forecast values, and a report. It shows
the approved scoring and produces the same route response format as the local replay.
Open the notebook from this repository and use a Python environment with the
dependencies listed in the cascading backend README.

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "backend/src/plexa").is_dir()),
    None,
)
if root is None:
    raise RuntimeError("Open the notebook from inside the Plexa repository.")
sys.path.insert(0, str(root / "backend/src"))
from plexa.cascading import route_request

nodes = pd.MultiIndex.from_tuples(
    [("0000001", "M"), ("0000002", "M"), ("0000003", "M")],
    names=["node_id", "node_level"],
)
points = [(588000, 4509000), (588050, 4509037.5), (588100, 4509000)]
node_positions = pd.DataFrame(points, columns=["x_m", "y_m"])
roads = gpd.GeoDataFrame({
    "source_fid": [1, 2, 3, 4],
    "Street": ["Demo Street", "Demo Bypass A", "Demo Bypass B", "Demo Private Lane"],
    "FeatureTyp": ["0", "0", "0", "6"],
    "RW_TYPE": ["1", "1", "1", "1"],
}, geometry=[
    LineString([points[0], points[2]]),
    LineString([points[0], points[1]]),
    LineString([points[1], points[2]]),
    LineString([points[0], points[2]]),
], crs="EPSG:32618")
arcs = pd.DataFrame({
    "source_fid": [1, 2, 3, 4],
    "lion_segment_id": ["DEMO-DIRECT", "DEMO-BYPASS-A", "DEMO-BYPASS-B", "DEMO-PRIVATE"],
    "from_node_index": [0, 0, 1, 0],
    "to_node_index": [2, 1, 2, 2],
    "travel_sense": ["from_to"] * 4,
    "length_m": roads.geometry.length,
    "mobility_p50": [0.0, 0.0, None, 0.0],
    "climate_reported_flooding_score": [0.001, 0.002, None, None],
    "climate_model_alert": pd.Series([False, False, None, None], dtype="boolean"),
})
warning_links = pd.DataFrame({
    "incident_id": ["demo:road-warning-001"],
    "source_fid": [1],
    "result.rule_id": ["S311-09"],
})
climate = pd.DataFrame({"climate_status": ["ok", "ok", "unavailable"]})

def load_demo_hour(clock):
    return {"arcs": arcs, "warning_links": warning_links, "climate": climate}

request = {
    "scenario": "ida",
    "as_of_utc": "2021-09-02T02:00:00Z",
    "origin": {"node_id": "0000001", "node_level": "M"},
    "destination": {"node_id": "0000003", "node_level": "M"},
}
route_inputs = {
    "load_hour": load_demo_hour,
    "nodes": nodes,
    "roads": roads,
    "node_positions": node_positions,
}
response = route_request(request, **route_inputs)
route_json = json.dumps(response, allow_nan=False, indent=2)

In [ ]:
print(f"Scenario cutoff {response['as_of_utc']}")
for label, route in response["routes"].items():
    print(
        f"{label}  {route['distance_m']:.1f} m, "
        f"score {route['routing_cost_m']:.2f}, "
        f"traffic coverage {route['traffic_coverage']['percent']:.0f}%"
    )
print("Route changed:", response["changed_from_shortest"])

In [ ]:
cleared = route_request(
    request, **route_inputs, cleared_incident_ids=["demo:road-warning-001"]
)
closed = route_request(request, **route_inputs, closed_source_fids=[1])
assert response["routes"]["shortest"]["distance_m"] == 100.0
assert response["routes"]["shortest"]["penalties_m"]["road_warning"] == 175.0
assert response["routes"]["recommended"]["distance_m"] == 125.0
assert cleared["routes"]["recommended"]["distance_m"] == 100.0
assert closed["routes"]["shortest"]["distance_m"] == 125.0